# Millionaire Prediction from Habits Using a Neural Network (MLP)

**Group Mini Project - Deep Learning (Binary Classification)**

---

## 1. Problem Case

This project builds a binary classification model with a Neural Network (MLP) to predict whether an individual is a **millionaire** from personal characteristics and lifestyle/behavioural habits, without direct monetary or balance-sheet inputs.

| Value | Class |
| :---: | :--- |
| `0` | Not Millionaire |
| `1` | Millionaire |

### Target definition

The target is **millionaire status**, defined as:

```python
df["is_millionaire"] = (df["net_worth"] >= 1_000_000).astype(int)
```

A billionaire target (`net_worth >= 1_000_000_000`) is not trainable on this dataset because there are no positive examples; the maximum net worth is approximately $105.8M. The notebook therefore evaluates the valid millionaire target and validates it against the dataset's `millionaire_status` label.

### Data leakage policy

The target is derived from `net_worth`. Monetary columns, asset values, liabilities, financial ratios, engineered wealth metrics, and target-derived flags are excluded from the primary feature set so the model cannot read the answer from the balance sheet.

The primary feature set retains personal, lifestyle, and behavioural variables. Potential socioeconomic proxy features are documented and evaluated separately; they are not treated as direct target leakage.

---

## 2. Research Questions

**Main RQ:** Can a neural network predict millionaire status from personal characteristics and lifestyle/behavioural habits without direct monetary or wealth-balance-sheet information?

**Additional RQs**
1. How accurately can the network separate the two millionaire-status classes using leakage-controlled features?
2. Which neural-network architecture gives the best performance?
3. How do different feature groups (personal vs. lifestyle) affect performance?
4. How does hyperparameter/optimiser tuning affect performance?
5. How much does performance change after removing potential socioeconomic proxy features?

## 4. Setup & Imports

We fix all random seeds for reproducibility and load the standard deep-learning stack
(TensorFlow / Keras, scikit-learn, pandas, matplotlib, seaborn).

In [ ]:
import os, sys, time, json, random, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONHASHSEED"] = "42"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

# ----------------------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"

DATA_DIR = "data"
OUT_DIR  = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("Python      :", sys.version.split()[0])
print("TensorFlow  :", tf.__version__)
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("GPU devices :", tf.config.list_physical_devices("GPU"))

## 5. Load the Dataset

We load the full **Millionaire Habits Dataset** (100,000 synthetic individuals × 274 columns) together
with its **data dictionary**, which we will reuse later to build the *feature groups* automatically.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Millionaire_Habits_Dataset.csv"))
data_dict = pd.read_csv(os.path.join(DATA_DIR, "Millionaire_Habits_Data_Dictionary.csv"))

print("Dataset shape      :", df.shape)
print("Data dictionary    :", data_dict.shape)
print()
print("Column dtypes:")
print(df.dtypes.value_counts().to_string())
df.head(3)

In [ ]:
# Category breakdown of the 274 columns, taken from the data dictionary
cat_summary = (data_dict.groupby("category")["column_name"].count()
               .rename("n_columns").to_frame())
cat_summary

## 6. Target Variable Construction

### 6.1 Why the notebook evaluates millionaire status

We first confirm the finding that motivated our adaptation: the dataset contains **no billionaires**.

In [ ]:
nw = df["net_worth"]
print("net_worth summary (USD):")
print(nw.describe().apply(lambda v: f"{v:,.2f}").to_string())
print()
print("Positive-class counts for candidate wealth thresholds:")
rows = []
for t in [1_000_000, 5_000_000, 10_000_000, 50_000_000, 100_000_000, 1_000_000_000]:
    c = int((nw >= t).sum())
    rows.append({"threshold": f"${t:,.0f}", "n_positive": c, "pct": round(c / len(df) * 100, 4)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print(">>> net_worth >= $1,000,000,000 yields", int((nw >= 1e9).sum()),
      "positives -> the literal billionaire target would be EMPTY.")
print(">>> We therefore use the $1,000,000 (millionaire) threshold.")


### 6.2 Chosen target

The notebook evaluates the dataset's validated **millionaire** definition because the literal `$1B` rule has zero positive examples. This is a documented dataset adaptation, not a claim that millionaire and billionaire are equivalent targets.

In [ ]:
THRESHOLD = 1_000_000  # $1,000,000
df["is_millionaire"] = (df["net_worth"] >= THRESHOLD).astype(int)

counts = df["is_millionaire"].value_counts().sort_index()
pct    = df["is_millionaire"].value_counts(normalize=True).sort_index() * 100
print("Target distribution:")
for k in counts.index:
    label = "Millionaire (1)" if k == 1 else "Not Millionaire (0)"
    print(f"  {label:<20s}: {counts[k]:>6,}  ({pct[k]:.2f}%)")

# Sanity check: our target matches the dataset's own millionaire_status column
print("\nAgreement with dataset 'millionaire_status':",
      f"{(df['is_millionaire'] == df['millionaire_status']).mean():.4f}")

## 7. Exploratory Data Analysis (EDA)

### 7.1 Class balance and net-worth distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) class balance
ax = axes[0]
sns.countplot(x="is_millionaire", data=df, ax=ax, palette=["#e07a5f", "#3d9970"])
ax.set_xticklabels(["Not Millionaire\n(0)", "Millionaire\n(1)"])
ax.set_title("Target Class Balance"); ax.set_xlabel(""); ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.12)

# (b) net worth distribution (log scale)
ax = axes[1]
sns.histplot(df["net_worth"].clip(lower=1), bins=80, log_scale=(True, False),
             color="#3d9970", ax=ax)
ax.axvline(THRESHOLD, color="crimson", ls="--", lw=2, label=f"Threshold = ${THRESHOLD:,.0f}")
ax.set_title("Net Worth Distribution (log-x)"); ax.set_xlabel("Net Worth (USD, log)"); ax.legend()

# (c) wealth bracket composition
ax = axes[2]
order = ["negative_net_worth","zero_to_10k","10k_to_50k","50k_to_100k","100k_to_250k",
         "250k_to_500k","500k_to_1m","1m_to_2.5m","2.5m_to_5m","5m_to_10m","10m_plus"]
order = [o for o in order if o in df["wealth_category"].unique()]
sns.countplot(y="wealth_category", data=df, order=order, ax=ax, color="#5276a7")
ax.axhline(len(order) - order.index("1m_to_2.5m") - 0.5, color="crimson", ls="--", lw=1.5)
ax.set_title("Wealth Bracket Composition"); ax.set_xlabel("Count"); ax.set_ylabel("")

plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "eda_target.png"), dpi=120, bbox_inches="tight")
plt.show()

print("Class balance is fairly balanced (57% / 43%) at the $1M threshold — a useful property we revisit "
      "in the class-imbalance experiment.")

### 7.2 Missing values

The dataset intentionally contains ~1–5% realistic missingness in a subset of columns.

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"Columns with missing values: {len(miss)} / {df.shape[1]}")
print(f"Average missingness across those columns: {miss.mean()*100:.2f}%")
print()
print("Top 20 most-missing columns:")
print((miss.head(20) * 100).round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
(miss.head(20) * 100)[::-1].plot(kind="barh", ax=ax, color="#c1666b")
ax.set_title("Top 20 Columns by Missing Rate (%)"); ax.set_xlabel("% missing")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "eda_missing.png"), dpi=120, bbox_inches="tight")
plt.show()

### 7.3 Which features correlate with the target?

We compute the point-biserial correlation of every **numeric** feature with `is_millionaire`
and inspect the strongest positive / negative signals.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
corr = (df[num_cols].corrwith(df["is_millionaire"])
        .drop(labels=["is_millionaire", "millionaire_status", "multi_millionaire_status",
                      "net_worth", "net_worth_log", "net_worth_percentile"], errors="ignore")
        .sort_values())

top_pos = corr.tail(15)
top_neg = corr.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_pos[::-1].plot(kind="barh", ax=axes[0], color="#3d9970")
axes[0].set_title("Top 15 Positive Correlations with Target"); axes[0].set_xlabel("Correlation")
top_neg.plot(kind="barh", ax=axes[1], color="#c1666b")
axes[1].set_title("Top 15 Negative Correlations with Target"); axes[1].set_xlabel("Correlation")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "eda_corr.png"), dpi=120, bbox_inches="tight")
plt.show()

print("Note the leakage risk: the strongest correlates (net-worth percentiles, asset values, "
      "investment-income ratios) are direct components of net worth itself.")

## 8. Feature Engineering — Leakage-Free Feature Policy

The target is `is_millionaire = (net_worth >= 1_000_000)`. Any column that **is** net worth, a
**monetary amount / balance**, a **financial ratio / percentage**, a **percentile**, or a
**target-derived flag** would let the model "read the answer" instead of learning from habits.

We therefore adopt a strict **whitelist** policy:

> **Keep ONLY** genuine personal characteristics (age, education, occupation, marital status, …)
> and lifestyle / behavioural habits (exercise, reading, sleep, work, routines, time management,
> and *behavioural* financial habits such as budgeting or investing **frequency**).
>
> **Exclude EVERYTHING monetary or wealth-derived**: incomes, salaries, balances, asset values,
> savings / investment **amounts**, debt amounts, all financial ratios and percentages, percentiles,
> wealth brackets, and every engineered / target-derived column.

| Bucket | Action | Why |
| --- | --- | --- |
| Identifiers / meta | Drop | No generalisable signal (`synthetic_person_id`, seeds, currency) |
| Target & wealth flags | Drop | Directly derived from the target (`millionaire_status`, `wealth_category`, …) |
| Net-worth components | Drop | Assets/liabilities sum to net worth → answer leakage |
| Monetary amounts | Drop | Income / savings / debt amounts are an *outcome* of wealth, not a habit |
| Financial ratios & % | Drop | Computed from monetary columns → indirect leakage |
| Wealth-proxy scores | Drop | `financial_buffer_score`, `debt_free_status`, `emergency_fund_months` track wealth |
| Redundant / constant | Drop | `age_group` (≡ age), `highest_degree` (≡ education_level), `has_debt` (constant) |

The machine-readable exclusion table (column → reason) is produced below and saved to
`outputs/excluded_columns.csv`; the retained whitelist is saved to `outputs/retained_features.csv`.


In [ ]:
# =====================================================================
# A) RETAINED WHITELIST  (leakage-free: personal + lifestyle/behaviour)
# =====================================================================
PERSONAL_FEATURES = [
    # --- A. Identification / meta (NO ids, NO derived age_group) ---
    "country", "region", "urban_rural", "age", "gender", "marital_status", "dependents_count",
    # --- B. Education (highest_degree is a duplicate of education_level -> dropped) ---
    "education_level", "field_of_study", "education_years", "financial_education_level",
    "self_learning_hours_per_month", "books_read_per_year", "nonfiction_books_per_year",
    "business_books_per_year", "finance_books_per_year", "online_courses_per_year",
    "mentorship_access",
    # --- C. Career  (NON-financial only) ---
    "employment_status", "occupation_category", "industry", "years_of_experience",
    "career_changes_count", "current_job_tenure_years", "income_stability_score",
    "job_satisfaction", "career_satisfaction", "remote_work_frequency", "weekly_work_hours",
    "leadership_role", "management_level", "professional_network_score",
    # --- D. Entrepreneurship  (NON-financial only) ---
    "business_owner", "business_count", "business_age_years", "startup_experience",
    "entrepreneurship_experience_years", "side_business", "business_failure_count",
    "business_risk_tolerance", "entrepreneurial_intention", "innovation_score",
]

LIFESTYLE_FEATURES = [
    # --- F. Saving habits (behavioural only) ---
    "automatic_savings", "budgeting_frequency", "expense_tracking_frequency",
    "financial_goal_setting", "financial_goal_review_frequency", "impulse_purchase_frequency",
    "large_purchase_planning", "needs_vs_wants_score", "lifestyle_inflation_score",
    "frugality_score",
    # --- G. Investing habits (behavioural only) ---
    "invests_regularly", "investment_frequency", "investment_horizon_years", "index_fund_usage",
    "individual_stock_usage", "real_estate_investing", "business_investing",
    "international_investing", "crypto_exposure", "investment_risk_tolerance",
    "investment_knowledge", "financial_advisor_usage", "portfolio_review_frequency",
    "panic_selling_frequency", "dollar_cost_averaging", "long_term_investing_score",
    # --- H. Debt behaviour (behavioural only) ---
    "credit_card_usage", "credit_card_payment_behavior", "debt_repayment_strategy",
    "debt_stress_score",
    # --- I. Productivity ---
    "daily_productivity_score", "weekly_productivity_score", "morning_routine_score",
    "evening_routine_score", "daily_planning", "weekly_planning", "monthly_goal_review",
    "deep_work_hours_per_day", "focused_work_sessions", "distraction_frequency",
    "social_media_hours_per_day", "screen_time_hours_per_day", "task_completion_rate",
    "procrastination_score", "time_management_score", "priority_setting_score",
    "calendar_usage", "to_do_list_usage", "goal_tracking_frequency",
    # --- J. Time management ---
    "wake_up_time", "sleep_time", "sleep_duration_hours", "work_start_time", "work_end_time",
    "commute_hours", "family_time_hours", "learning_hours_per_week", "exercise_hours_per_week",
    "leisure_hours_per_week", "planning_minutes_per_day", "unproductive_hours_per_day",
    # --- K. Health & wellness ---
    "exercise_days_per_week", "strength_training_days", "cardio_days", "walking_steps_per_day",
    "healthy_meal_frequency", "fast_food_frequency", "vegetable_intake_frequency",
    "water_intake_liters", "sleep_quality_score", "stress_level", "meditation_frequency",
    "mindfulness_score", "energy_level", "health_routine_consistency",
    # --- L. Psychology / behaviour ---
    "financial_discipline_score", "self_control_score", "delayed_gratification_score",
    "risk_tolerance_score", "risk_aversion_score", "confidence_score", "resilience_score",
    "growth_mindset_score", "long_term_orientation_score", "goal_orientation_score",
    "consistency_score", "adaptability_score", "decision_making_score", "financial_anxiety_score",
    "stress_management_score",
    # --- M. Social / networking ---
    "networking_frequency", "professional_events_per_year", "mentor_relationship",
    "mentor_frequency", "peer_financial_discussion_frequency", "community_involvement",
    "social_support_score", "professional_network_size", "high_income_peer_exposure",
    "business_network_score",
    # --- N. Lifestyle (NON-monetary only) ---
    "housing_type", "home_ownership", "luxury_purchase_frequency", "travel_frequency_per_year",
    "car_ownership", "vehicle_count", "minimalism_score", "lifestyle_satisfaction",
    # --- O. Financial behaviour (behavioural only) ---
    "financial_literacy_score", "financial_planning_score", "net_worth_tracking_frequency",
    "investment_research_hours_per_month", "tax_planning_behavior", "insurance_coverage_score",
    "retirement_planning_score", "estate_planning_status", "financial_goal_count",
    "short_term_goal_count", "long_term_goal_count", "financial_review_frequency",
    "money_mindset_score",
    # --- P. Habit consistency ---
    "habit_tracking", "habit_tracking_frequency", "routine_consistency_score",
    "saving_consistency_score", "investment_consistency_score", "exercise_consistency_score",
    "sleep_consistency_score", "learning_consistency_score", "goal_consistency_score",
    "budget_consistency_score", "overall_habit_consistency_score",
]

# =====================================================================
# B) EXPLICIT EXCLUSION LISTS  (column -> reason)
# =====================================================================
EXCLUDE_ID_META = [
    "synthetic_person_id", "dataset_version", "generation_seed", "currency",
    "exchange_rate_to_usd",
]
EXCLUDE_TARGET_FLAGS = [
    "millionaire_status", "multi_millionaire_status", "wealth_bracket", "wealth_category",
    "financial_independence_status", "high_saver_status", "high_investor_status",
    "entrepreneur_status", "high_financial_literacy", "high_productivity", "high_debt_risk",
    "financial_stability_score",
]
EXCLUDE_NETWORTH = ["net_worth", "net_worth_log", "net_worth_percentile"]
EXCLUDE_WEALTH_COMPONENTS = [
    "total_assets", "cash_savings", "checking_savings", "investment_assets", "stocks_value",
    "bonds_value", "real_estate_value", "retirement_accounts", "business_equity",
    "alternative_assets", "vehicle_assets", "other_assets", "total_liabilities",
    "mortgage_debt", "student_debt", "credit_card_debt", "business_debt", "other_debt",
    "total_debt",
]
EXCLUDE_MONETARY_AMOUNTS = [
    "annual_salary", "annual_bonus", "other_income", "total_annual_income", "income_percentile",
    "monthly_income", "monthly_expenses", "monthly_savings", "annual_savings",
    "monthly_investment_amount", "annual_investment_amount", "business_revenue",
    "business_profit", "side_business_income", "passive_business_income", "employees_count",
    "credit_card_balance", "mortgage_payment", "loan_payment", "high_interest_debt",
]
EXCLUDE_LIFESTYLE_COSTS = [
    "annual_housing_cost", "annual_transport_cost", "annual_food_cost",
    "annual_entertainment_cost", "annual_travel_cost", "annual_health_cost",
    "annual_education_cost", "annual_luxury_spending", "annual_charity_donations",
]
EXCLUDE_FINANCIAL_RATIOS = [
    "savings_rate", "business_profit_margin", "debt_to_income_ratio", "credit_utilization",
    "retirement_investment_rate", "investment_rate", "housing_cost_ratio",
    "luxury_spending_ratio", "financial_buffer_months", "asset_diversification_score",
    "income_diversification_score", "passive_income_ratio", "business_income_ratio",
    "investment_income_ratio", "portfolio_diversification_score", "stock_allocation",
    "bond_allocation", "real_estate_allocation", "cash_allocation", "alternative_asset_allocation",
    "work_to_leisure_ratio", "learning_to_work_ratio", "health_consistency_index",
    "financial_discipline_index", "wealth_building_index", "wealth_building_score",
    "emergency_fund_months",
]
EXCLUDE_WEALTH_PROXY = ["debt_free_status", "financial_buffer_score"]
EXCLUDE_REDUNDANT = ["age_group", "highest_degree"]
EXCLUDE_CONSTANT = ["has_debt"]

EXCLUDE_REASON = {}
for _cols, _reason in [
    (EXCLUDE_ID_META,            "Identifier / meta - no generalisable signal"),
    (EXCLUDE_TARGET_FLAGS,       "Target-derived flag (leaks the label)"),
    (EXCLUDE_NETWORTH,           "Direct net-worth value / transform"),
    (EXCLUDE_WEALTH_COMPONENTS,  "Wealth component (sums to net worth)"),
    (EXCLUDE_MONETARY_AMOUNTS,   "Monetary amount (income / savings / debt / revenue)"),
    (EXCLUDE_LIFESTYLE_COSTS,    "Monetary lifestyle cost (annual spend)"),
    (EXCLUDE_FINANCIAL_RATIOS,   "Financial ratio / percentage (derived from money)"),
    (EXCLUDE_WEALTH_PROXY,       "Wealth-proxy score / flag"),
    (EXCLUDE_REDUNDANT,          "Redundant duplicate of another retained feature"),
    (EXCLUDE_CONSTANT,           "Constant column (zero variance)"),
]:
    for _c in _cols:
        EXCLUDE_REASON[_c] = _reason

RETAINED_FEATURES = PERSONAL_FEATURES + LIFESTYLE_FEATURES

print("Leakage-free whitelist:")
print(f"  Personal characteristics : {len(PERSONAL_FEATURES):>3d}")
print(f"  Lifestyle & behavioural  : {len(LIFESTYLE_FEATURES):>3d}")
print(f"  => Retained total        : {len(RETAINED_FEATURES):>3d}")
print(f"  Explicitly excluded      : {len(EXCLUDE_REASON):>3d}  (by reason)")


### 8.1 Building the clean feature set + safety net

We derive the retained set from the whitelist and the excluded set as *everything else*. A
**name-based safety net** then scans the retained set for any monetary-sounding column, so that a
column cannot slip through just because it was listed in the whitelist by mistake. The two known
intentional cases (`income_stability_score`, `high_income_peer_exposure`) are **scores**, not
amounts, and are confirmed as behavioural/social signals.


In [ ]:
CLEAN_FEATURES = [c for c in RETAINED_FEATURES if c in df.columns]
EXCLUDED_FEATURES = [c for c in df.columns if c not in CLEAN_FEATURES]

print(f"Total dataset columns            : {df.shape[1]}")
print(f"Retained (leakage-free) features : {len(CLEAN_FEATURES)}")
print(f"Excluded features                : {len(EXCLUDED_FEATURES)}")

# ---- name-based safety net -------------------------------------------
MONEY_HINTS = ["income", "salary", "bonus", "savings", "expense", "revenue", "profit",
               "debt", "loan", "mortgage", "asset", "value", "balance", "cost", "amount",
               "net_worth", "wealth", "allocation", "rate", "percentile", "margin", "payment",
               "equity", "spend", "fund"]
suspects = [c for c in CLEAN_FEATURES if any(h in c for h in MONEY_HINTS)]
print("\nSafety-net scan of retained features (monetary-sounding names):")
for c in suspects:
    print(f"  - {c:<32s} -> retained (score/behaviour, NOT a monetary amount)")

# ---- build the audit table -------------------------------------------
audit = pd.DataFrame({
    "column": EXCLUDED_FEATURES,
    "reason": [EXCLUDE_REASON.get(c, "Not on the leakage-free whitelist") for c in EXCLUDED_FEATURES],
})
audit = audit.sort_values(["reason", "column"]).reset_index(drop=True)
print("\nExclusion reasons (counts):")
print(audit["reason"].value_counts().to_string())
print("\nFirst 15 excluded columns:")
print(audit.head(15).to_string(index=False))

    # Persist the feature audit so the leakage decision is reviewable outside the notebook.
category_by_column = dict(zip(data_dict["column_name"], data_dict["category"]))
feature_audit = pd.DataFrame({
    "column": list(df.columns),
    "category": [category_by_column.get(c, "unknown") for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns],
    "action": ["KEEP" if c in CLEAN_FEATURES else "DROP" for c in df.columns],
    "reason": ["Leakage-free whitelist" if c in CLEAN_FEATURES else EXCLUDE_REASON.get(c, "Not on the leakage-free whitelist") for c in df.columns],
})
feature_audit.to_csv(os.path.join(OUT_DIR, "feature_audit.csv"), index=False)
assert "is_millionaire" not in CLEAN_FEATURES
assert not set(CLEAN_FEATURES).intersection({"net_worth", "millionaire_status", "total_assets", "total_liabilities"})
print(f"\nFeature audit saved to {os.path.join(OUT_DIR, 'feature_audit.csv')}")


In [ ]:
# Potential socioeconomic proxies: not direct leakage, but evaluated separately.
PROXY_FEATURES = [
    "home_ownership", "vehicle_count", "business_owner",
    "income_stability_score", "estate_planning_status",
    "insurance_coverage_score", "real_estate_investing",
    "financial_advisor_usage",
]
PROXY_FEATURES = [c for c in PROXY_FEATURES if c in CLEAN_FEATURES]
STRICT_BEHAVIOR_FEATURES = [c for c in CLEAN_FEATURES if c not in PROXY_FEATURES]
proxy_audit = pd.DataFrame({
    "feature": PROXY_FEATURES,
    "category": [category_by_column.get(c, "unknown") for c in PROXY_FEATURES],
    "status": "Potential socioeconomic proxy",
})
proxy_audit.to_csv(os.path.join(OUT_DIR, "proxy_features.csv"), index=False)
print(f"Potential proxy features retained in primary set: {len(PROXY_FEATURES)}")
print(f"Strict behaviour ablation feature count: {len(STRICT_BEHAVIOR_FEATURES)}")
proxy_audit


### 8.2 Feature groups (for Research Question 3)

We group the retained features into two interpretable, leakage-free groups using the data
dictionary's `category` field:

| Group | Dictionary categories |
| --- | --- |
| **Personal** | A. Identification/Meta (demographics) + B. Education + C. Career (non-financial) + D. Entrepreneurship (non-financial) |
| **Lifestyle & Behavioral** | F. Saving + G. Investing + H. Debt + I. Productivity + J. Time Mgmt + K. Health + L. Psychology + M. Social + N. Lifestyle + O. Financial Behavior + P. Habit Consistency |

Note: category **E. Wealth**, **Q. Target/ML** and **R. Engineered Features** are *entirely
excluded*, and monetary members of C/D/F/G/H/N/O are removed by the whitelist.


In [ ]:
cat_of = dict(zip(data_dict["column_name"], data_dict["category"]))

GROUP_CATS = {
    "Personal": ["A. Identification/Meta", "B. Education", "C. Career", "D. Entrepreneurship"],
    "Lifestyle & Behavioral": ["F. Saving Habits", "G. Investing Habits", "H. Debt Behavior",
                               "I. Productivity", "J. Time Management", "K. Health & Wellness",
                               "L. Psychology/Behavior", "M. Social/Networking", "N. Lifestyle",
                               "O. Financial Behavior", "P. Habit Consistency"],
}

FEATURE_GROUPS = {}
for g, cats in GROUP_CATS.items():
    cols = [c for c, cat in cat_of.items() if cat in cats]
    cols = [c for c in cols if c in CLEAN_FEATURES]      # whitelist already removes leakage
    FEATURE_GROUPS[g] = cols

for g, cols in FEATURE_GROUPS.items():
    print(f"{g:<24s}: {len(cols):>3d} columns")

_total = sum(len(v) for v in FEATURE_GROUPS.values())
print(f"\nSum of group sizes         : {_total}")
print(f"CLEAN_FEATURES size        : {len(CLEAN_FEATURES)}")
print(f"Difference (outside groups): {len(CLEAN_FEATURES) - _total}")


In [ ]:
# Which clean features fall outside the two groups? (should be none)
grouped = set(sum(FEATURE_GROUPS.values(), []))
ungrouped = [c for c in CLEAN_FEATURES if c not in grouped]
print("Clean features not assigned to a group:", len(ungrouped))
print(ungrouped[:20])


In [ ]:
# =====================================================================
# Class-distribution report  (target = is_millionaire)
# =====================================================================
counts = df["is_millionaire"].value_counts().sort_index()
pcts   = df["is_millionaire"].value_counts(normalize=True).sort_index() * 100
class_distribution = pd.DataFrame({
    "class": ["Not Millionaire (0)", "Millionaire (1)"],
    "count": [int(counts.get(0, 0)), int(counts.get(1, 0))],
    "percent": [round(float(pcts.get(0, 0)), 2), round(float(pcts.get(1, 0)), 2)],
})
print("=== Target class distribution (whole dataset) ===")
print(class_distribution.to_string(index=False))
print(f"\nBase rate (positive)  : {df['is_millionaire'].mean():.4f}")
print(f"Majority-class share  : {max(pcts)/100:.4f}  <- a trivial 'always predict majority' baseline")

# =====================================================================
# Retained (leakage-free) features, with meanings
# =====================================================================
desc_of = data_dict.set_index("column_name")["description"].to_dict()
print("\n=== Retained leakage-free features ===")
for g, cols in FEATURE_GROUPS.items():
    print(f"\n[{g}]  ({len(cols)} features)")
    for c in cols:
        print(f"   {c:<36s} | {desc_of.get(c, '')[:60]}")


## 9. Data Splitting (Stratified 70 / 15 / 15)

We split **before** any preprocessing so that imputation/scaling statistics are learned on the
training set only — this prevents information from the validation/test sets leaking into training.
The split is **stratified** to preserve the class proportion in every subset.

In [ ]:
X_all = df[CLEAN_FEATURES]
y_all = df["is_millionaire"].values

idx = np.arange(len(df))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=SEED, stratify=y_all)
val_idx,  test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED, stratify=y_all[temp_idx])

y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]

print(f"Train : {len(train_idx):>6,} rows  (positive rate {y_train.mean():.4f})")
print(f"Val   : {len(val_idx):>6,} rows  (positive rate {y_val.mean():.4f})")
print(f"Test  : {len(test_idx):>6,} rows  (positive rate {y_test.mean():.4f})")
print(f"Total : {len(df):>6,} rows")

## 10. Preprocessing Pipeline

* **Numeric features** → median imputation → standard scaling
* **Categorical features** → most-frequent imputation → one-hot encoding (`handle_unknown="ignore"`)

The pipeline is **fit on the training split only** and then applied to validation and test.

In [ ]:
def build_matrices(features, verbose=True):
    '''Fit a preprocessing pipeline on the train split and transform train/val/test.'''
    num = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in features if not pd.api.types.is_numeric_dtype(df[c])]

    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc",  StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ])
    Xtr = pre.fit_transform(df.iloc[train_idx][features]).astype("float32")
    Xva = pre.transform(df.iloc[val_idx][features]).astype("float32")
    Xte = pre.transform(df.iloc[test_idx][features]).astype("float32")
    if verbose:
        print(f"  numeric={len(num):>3d}  categorical={len(cat):>3d}  ->  encoded dims={Xtr.shape[1]}")
    return Xtr, Xva, Xte, pre

print("Full clean feature set:")
X_train, X_val, X_test, PRE = build_matrices(CLEAN_FEATURES)
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

## 11. Model Builder & Experiment Runner

We define reusable helpers:

* `build_mlp` — builds a configurable MLP (`Dense→ReLU→Dropout` stack → `Sigmoid` output).
* `random_oversample` — naive random oversampling of the minority class (training set only).
* `compute_metrics` — accuracy, precision, recall, F1, ROC-AUC, PR-AUC and the confusion-matrix cells.
* `run_experiment` — trains one configuration with early stopping and returns validation + test metrics.

**Training protocol** (as per the brief): Binary Cross-Entropy loss, Adam optimiser, ReLU hidden
activations, Sigmoid output, Early Stopping on `val_loss`, and a validation set for monitoring.

In [ ]:
def build_mlp(n_features, hidden=(64, 32), dropout=0.3, l2=0.0, seed=SEED):
    # Build a fresh MLP from scratch (never reuses previous weights).
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    reg = regularizers.l2(l2) if l2 > 0 else None
    model = models.Sequential([layers.Input(shape=(n_features,))])
    for h in hidden:
        model.add(layers.Dense(h, activation="relu", kernel_regularizer=reg))
        if dropout and dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="sigmoid"))
    return model


def random_oversample(X, y, seed=SEED):
    # Random oversampling of the minority class (TRAINING data only).
    rng = np.random.RandomState(seed)
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    n = max(len(i0), len(i1))
    i0s = rng.choice(i0, n, replace=len(i0) < n)
    i1s = rng.choice(i1, n, replace=len(i1) < n)
    sel = np.concatenate([i0s, i1s]); rng.shuffle(sel)
    return X[sel], y[sel]


def compute_metrics(y_true, y_prob, thr=0.5, prefix=""):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        f"{prefix}accuracy":  accuracy_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred, zero_division=0),
        f"{prefix}recall":    recall_score(y_true, y_pred, zero_division=0),
        f"{prefix}f1":        f1_score(y_true, y_pred, zero_division=0),
        f"{prefix}roc_auc":   roc_auc_score(y_true, y_prob),
        f"{prefix}pr_auc":    average_precision_score(y_true, y_prob),
        f"{prefix}tn": tn, f"{prefix}fp": fp, f"{prefix}fn": fn, f"{prefix}tp": tp,
    }


RESULTS = []   # collect every experiment here


def make_optimizer(name, lr):
    if name == "adam":
        return tf.keras.optimizers.Adam(learning_rate=lr)
    if name == "rmsprop":
        return tf.keras.optimizers.RMSprop(learning_rate=lr)
    raise ValueError(f"Unknown optimizer: {name}")


def run_experiment(name, Xtr, ytr, Xva, yva, Xte, yte,
                   hidden=(64, 32), dropout=0.3, lr=1e-3, batch_size=256,
                   epochs=40, patience=6, class_weight=None, oversample=False,
                   l2=0.0, optimizer="adam", record=True, verbose=0):
    # Train one MLP from scratch and evaluate on val/test.
    Xtr_use, ytr_use = (random_oversample(Xtr, ytr) if oversample else (Xtr, ytr))
    model = build_mlp(Xtr.shape[1], hidden=hidden, dropout=dropout, l2=l2)
    model.compile(optimizer=make_optimizer(optimizer, lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    es = callbacks.EarlyStopping(monitor="val_loss", patience=patience,
                                 restore_best_weights=True, verbose=0)
    t0 = time.time()
    hist = model.fit(Xtr_use, ytr_use, validation_data=(Xva, yva),
                     epochs=epochs, batch_size=batch_size,
                     class_weight=class_weight, callbacks=[es], verbose=verbose)
    train_time = time.time() - t0

    p_val  = model.predict(Xva,  verbose=0).ravel()
    p_test = model.predict(Xte,  verbose=0).ravel()

    row = {"name": name, "hidden": str(list(hidden)), "dropout": dropout, "lr": lr,
           "batch_size": batch_size, "optimizer": optimizer, "oversample": oversample,
           "class_weight": bool(class_weight), "epochs_run": len(hist.history["loss"]),
           "train_time_s": round(train_time, 1)}
    row.update(compute_metrics(yva,  p_val,  prefix="val_"))
    row.update(compute_metrics(yte,  p_test, prefix="test_"))
    if record:
        RESULTS.append(row)
    return model, hist, row, p_val, p_test


print("Helpers ready (build_mlp / run_experiment now support optimizer='adam'|'rmsprop').")


## 12. Experiment 1 — Baseline Model

```text
Input → Dense(32) → ReLU → Dense(1) → Sigmoid
```

The simplest possible MLP, used as the reference point for every later experiment.

In [ ]:
t0 = time.time()
base_model, base_hist, base_row, base_pval, base_ptest = run_experiment(
    "Exp1 - Baseline (32)", X_train, y_train, X_val, y_val, X_test, y_test,
    hidden=(32,), dropout=0.0, lr=1e-3, batch_size=32)
print(f"Baseline finished in {time.time()-t0:.1f}s  ({base_row['epochs_run']} epochs)")
print(f"  Val  -> acc={base_row['val_accuracy']:.4f}  f1={base_row['val_f1']:.4f}  auc={base_row['val_roc_auc']:.4f}")
print(f"  Test -> acc={base_row['test_accuracy']:.4f}  f1={base_row['test_f1']:.4f}  auc={base_row['test_roc_auc']:.4f}")


In [ ]:
def plot_history(hist, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    axes[0].plot(hist.history["loss"], label="train"); axes[0].plot(hist.history["val_loss"], label="val")
    axes[0].set_title(f"{title} — Loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE"); axes[0].legend()
    axes[1].plot(hist.history["accuracy"], label="train"); axes[1].plot(hist.history["val_accuracy"], label="val")
    axes[1].set_title(f"{title} — Accuracy"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("acc"); axes[1].legend()
    plt.tight_layout(); plt.show()

plot_history(base_hist, "Baseline (32)")

## 13. Experiment 2 — Network Architecture

We compare three depths (all with dropout 0.3):

| Model | Architecture |
| --- | --- |
| A | Input → 32 → Output |
| B | Input → 64 → 32 → Output |
| C | Input → 128 → 64 → 32 → Output |

In [ ]:
ARCH_CONFIGS = {
    "Exp2 · Model A (32)":        (32,),
    "Exp2 · Model B (64-32)":     (64, 32),
    "Exp2 · Model C (128-64-32)": (128, 64, 32),
}
arch_models = {}
for name, hidden in ARCH_CONFIGS.items():
    m, h, row, _, _ = run_experiment(name, X_train, y_train, X_val, y_val, X_test, y_test,
                                     hidden=hidden, dropout=0.3, lr=1e-3, batch_size=256)
    arch_models[name] = m
    print(f"{name:<28s} ep={row['epochs_run']:>2d}  "
          f"val_f1={row['val_f1']:.4f}  test_f1={row['test_f1']:.4f}  test_auc={row['test_roc_auc']:.4f}")

## 14. Experiment 3 — Dropout Regularisation

Using the best architecture from Experiment 2, we vary the dropout rate to study overfitting:
`{0.0, 0.2, 0.3, 0.5}`.

In [ ]:
best_arch_name = max(ARCH_CONFIGS, key=lambda k: next(r["val_f1"] for r in RESULTS if r["name"] == k))
best_hidden = ARCH_CONFIGS[best_arch_name]
print("Best architecture from Exp2:", best_arch_name, "->", best_hidden)

for dr in [0.0, 0.2, 0.3, 0.5]:
    m, h, row, _, _ = run_experiment(f"Exp3 · Dropout {dr}", X_train, y_train, X_val, y_val, X_test, y_test,
                                     hidden=best_hidden, dropout=dr, lr=1e-3, batch_size=256)
    gap = row["val_accuracy"] - row["test_accuracy"]
    print(f"dropout={dr:<4} ep={row['epochs_run']:>2d}  val_acc={row['val_accuracy']:.4f}  "
          f"test_acc={row['test_accuracy']:.4f}  test_f1={row['test_f1']:.4f}  (val-test gap {gap:+.4f})")

## 15. Experiment 4 — Hyperparameter Tuning

We tune the key hyperparameters requested in the brief. To keep the search tractable we use a
**staged grid search**, always selecting on the validation set:

* **Stage A** — Learning rate × Batch size
* **Stage B** — Architecture × Dropout rate (at the best learning-rate/batch-size from Stage A)

### 15.1 Stage A — Learning rate × Batch size

In [ ]:
t0 = time.time()
stageA = []
for lr in [1e-3, 5e-4, 1e-4]:
    for bs in [16, 32, 64]:
        m, h, row, _, _ = run_experiment(f"Exp4A - lr={lr} bs={bs}", X_train, y_train, X_val, y_val, X_test, y_test,
                                         hidden=best_hidden, dropout=0.3, lr=lr, batch_size=bs)
        stageA.append(row)
        print(f"lr={lr:<7} bs={bs:<4} ep={row['epochs_run']:>2d}  "
              f"val_f1={row['val_f1']:.4f}  val_auc={row['val_roc_auc']:.4f}")

bestA = max(stageA, key=lambda r: r["val_roc_auc"])
BEST_LR, BEST_BS = bestA["lr"], bestA["batch_size"]
print(f"\nStage A done in {time.time()-t0:.1f}s -> best lr={BEST_LR}, batch_size={BEST_BS} "
      f"(val_auc={bestA['val_roc_auc']:.4f})")


In [ ]:
# Visualise the Stage A grid
piv = pd.DataFrame(stageA).pivot_table(index="lr", columns="batch_size", values="val_roc_auc")
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.heatmap(piv, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax)
ax.set_title("Stage A — Validation ROC-AUC (LR × Batch size)")
plt.tight_layout(); plt.show()

### 15.2 Stage B — Architecture × Dropout rate

In [ ]:
t0 = time.time()
stageB = []
combos = [((32,), 0.0), ((64, 32), 0.2), ((64, 32), 0.4),
          ((128, 64, 32), 0.2), ((256, 128), 0.3), ((128, 64, 32), 0.4)]
for hidden, dr in combos:
    m, h, row, _, _ = run_experiment(f"Exp4B · {list(hidden)} dr={dr}", X_train, y_train, X_val, y_val, X_test, y_test,
                                     hidden=hidden, dropout=dr, lr=BEST_LR, batch_size=BEST_BS)
    stageB.append(row)
    print(f"{str(list(hidden)):<18s} dr={dr:<4} ep={row['epochs_run']:>2d}  "
          f"val_f1={row['val_f1']:.4f}  val_auc={row['val_roc_auc']:.4f}")

bestB = max(stageB, key=lambda r: r["val_roc_auc"])
print(f"\nStage B done in {time.time()-t0:.1f}s -> best = {bestB['name']} (val_auc={bestB['val_roc_auc']:.4f})")

In [ ]:
# Combine the tuning stages into a single table, sorted by validation ROC-AUC
tuned = pd.DataFrame(stageA + stageB)[
    ["name", "hidden", "dropout", "lr", "batch_size", "optimizer", "epochs_run",
     "val_accuracy", "val_precision", "val_recall", "val_f1", "val_roc_auc"]
].sort_values("val_roc_auc", ascending=False)
tuned.round(4).to_string(index=False)


### 15.3 Stage C - Optimiser (Adam vs. RMSprop)

The brief asks to compare optimisers. We hold the Stage-A/B winners fixed and swap only the
optimiser.


In [ ]:
t0 = time.time()
stageC = []
for opt in ["adam", "rmsprop"]:
    m, h, row, _, _ = run_experiment(f"Exp4C - optimizer={opt}", X_train, y_train, X_val, y_val, X_test, y_test,
                                     hidden=best_hidden, dropout=0.3, lr=BEST_LR, batch_size=BEST_BS,
                                     optimizer=opt)
    stageC.append(row)
    print(f"optimizer={opt:<8s} ep={row['epochs_run']:>2d}  val_f1={row['val_f1']:.4f}  "
          f"val_auc={row['val_roc_auc']:.4f}")

bestC = max(stageC, key=lambda r: r["val_roc_auc"])
print(f"\nStage C done in {time.time()-t0:.1f}s -> best optimizer = {bestC['optimizer']} "
      f"(val_auc={bestC['val_roc_auc']:.4f})")

pd.DataFrame(stageC)[["name", "optimizer", "lr", "batch_size", "epochs_run",
                      "val_accuracy", "val_f1", "val_roc_auc"]].round(4).to_string(index=False)


## 16. Experiment 5 — Feature-Group Ablation (RQ 3)

We re-train the tuned architecture on each leakage-free feature group to quantify how much each
contributes:

`Personal` · `Lifestyle & Behavioral` · `Combined`

Each configuration is re-preprocessed (so the encoded dimensions match the selected features).


In [ ]:
t0 = time.time()
ablation_sets = {
    "Personal only":                    ["Personal"],
    "Lifestyle & Behavioral only":      ["Lifestyle & Behavioral"],
    "Combined (Personal + Lifestyle)":  ["Personal", "Lifestyle & Behavioral"],
}

ablation_rows = []
ablation_matrix_cache = {}
for label, groups in ablation_sets.items():
    feats = sum((FEATURE_GROUPS[g] for g in groups), [])
    print(f"\n{label}  ({len(feats)} raw features)")
    Xtr, Xva, Xte, _ = build_matrices(feats, verbose=True)
    ablation_matrix_cache[label] = (Xtr, Xva, Xte)
    m, h, row, _, _ = run_experiment(f"Exp5 - {label}", Xtr, y_train, Xva, y_val, Xte, y_test,
                                     hidden=best_hidden, dropout=0.3, lr=BEST_LR, batch_size=BEST_BS)
    row["n_features_raw"] = len(feats); row["n_features_encoded"] = Xtr.shape[1]
    ablation_rows.append(row)
    print(f"  -> test_acc={row['test_accuracy']:.4f}  test_f1={row['test_f1']:.4f}  test_auc={row['test_roc_auc']:.4f}")

print(f"\nAblation done in {time.time()-t0:.1f}s")


In [ ]:
abl = pd.DataFrame(ablation_rows)[["name", "n_features_raw", "n_features_encoded",
                                   "test_accuracy", "test_precision", "test_recall",
                                   "test_f1", "test_roc_auc"]]
abl.round(4).to_string(index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_df = pd.DataFrame(ablation_rows)
x = np.arange(len(plot_df)); w = 0.27
ax.bar(x - w, plot_df["test_accuracy"],  w, label="Accuracy",  color="#5276a7")
ax.bar(x,      plot_df["test_f1"],        w, label="F1-Score",  color="#3d9970")
ax.bar(x + w,  plot_df["test_roc_auc"],   w, label="ROC-AUC",   color="#c1666b")
ax.set_xticks(x); ax.set_xticklabels(plot_df["name"].str.replace("Exp5 - ", ""), rotation=15, ha="right")
ax.set_ylim(0, 1.0); ax.set_title("Feature-Group Ablation - Test Performance"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "exp5_ablation.png"), dpi=120, bbox_inches="tight")
plt.show()


## 17. Experiment 6 — Class-Imbalance Handling

Although the `$1M` target is close to balanced (~57/43), the brief asks us to study imbalance
handling, so we compare three strategies on the same architecture:

1. **Baseline** (no reweighting)
2. **Class weighting** (balanced weights)
3. **Random oversampling** of the minority class (training set only)

In [ ]:
# Balanced class weights computed from the training split
n = len(y_train); n0 = int((y_train == 0).sum()); n1 = int((y_train == 1).sum())
class_weight = {0: n / (2 * n0), 1: n / (2 * n1)}
print("Computed class weights:", {k: round(v, 4) for k, v in class_weight.items()})

imbalance_cfgs = [
    ("Exp6 · No reweighting",  dict(class_weight=None,     oversample=False)),
    ("Exp6 · Class weights",   dict(class_weight=class_weight, oversample=False)),
    ("Exp6 · Oversampling",    dict(class_weight=None,     oversample=True)),
]
for name, kw in imbalance_cfgs:
    m, h, row, _, _ = run_experiment(name, X_train, y_train, X_val, y_val, X_test, y_test,
                                     hidden=best_hidden, dropout=0.3, lr=BEST_LR, batch_size=BEST_BS, **kw)
    print(f"{name:<26s} test -> acc={row['test_accuracy']:.4f}  prec={row['test_precision']:.4f}  "
          f"rec={row['test_recall']:.4f}  f1={row['test_f1']:.4f}  auc={row['test_roc_auc']:.4f}")

## 18. Full Experiment Comparison

We assemble every experiment into one table (the format requested in the brief) and identify the
**final model** using validation performance.

In [ ]:
res = pd.DataFrame(RESULTS)
cols = ["name", "hidden", "dropout", "lr", "batch_size",
        "val_accuracy", "val_precision", "val_recall", "val_f1", "val_roc_auc",
        "test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]
summary = res[cols].copy()
summary.round(4).to_string(index=False)

In [ ]:
# Headline comparison table (the "Expected Experiment Results" format from the brief)
headline = summary[[
    "name", "test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"
]].rename(columns={
    "name": "Model", "test_accuracy": "Accuracy", "test_precision": "Precision",
    "test_recall": "Recall", "test_f1": "F1-Score", "test_roc_auc": "ROC-AUC",
}).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
headline.round(4)

In [ ]:
# Bar chart of the top models by test ROC-AUC
top = summary.sort_values("test_roc_auc", ascending=False).head(12)
fig, ax = plt.subplots(figsize=(11, 6))
order = top.iloc[::-1]
ax.barh(order["name"], order["test_roc_auc"], color="#5276a7")
ax.set_xlim(0, 1.0)
ax.set_title("Top 12 Experiments by Test ROC-AUC")
ax.set_xlabel("Test ROC-AUC")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "model_comparison.png"), dpi=120, bbox_inches="tight")
plt.show()


## 19. Final Model — Training & Test Evaluation

The **final model** is the configuration with the best **validation ROC-AUC**. We retrain it with a
slightly larger patience budget and evaluate it once on the untouched **test set**, reporting the
full metric suite: accuracy, precision, recall, F1, ROC-AUC, PR-AUC and the confusion matrix.

In [ ]:
best_cfg = max(RESULTS, key=lambda r: r["val_roc_auc"])
print("Selected final configuration (best validation ROC-AUC):")
print(json.dumps({k: best_cfg[k] for k in
                  ["name", "hidden", "dropout", "lr", "batch_size", "optimizer"]}, indent=2))

final_model, final_hist, final_row, p_val_final, p_test_final = run_experiment(
    "FINAL", X_train, y_train, X_val, y_val, X_test, y_test,
    hidden=eval(best_cfg["hidden"]), dropout=best_cfg["dropout"], lr=best_cfg["lr"],
    batch_size=best_cfg["batch_size"], optimizer=best_cfg.get("optimizer", "adam"),
    epochs=60, patience=10, record=False)

print(f"\nFINAL model trained ({final_row['epochs_run']} epochs, {final_row['train_time_s']}s)")
print("=" * 60)
print("TEST SET METRICS (leakage-free)")
print("=" * 60)
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"  {k:<10s}: {final_row['test_'+k]:.4f}")
print(f"  confusion  : TN={final_row['test_tn']}  FP={final_row['test_fp']}  "
      f"FN={final_row['test_fn']}  TP={final_row['test_tp']}")


In [ ]:
plot_history(final_hist, "Final Model")

In [ ]:
y_pred_final = (p_test_final >= 0.5).astype(int)
print(classification_report(y_test, y_pred_final,
                            target_names=["Not Millionaire (0)", "Millionaire (1)"], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

# (a) Confusion matrix
cm = confusion_matrix(y_test, y_pred_final, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=axes[0], cbar=False,
            xticklabels=["Not Mil.", "Millionaire"], yticklabels=["Not Mil.", "Millionaire"])
axes[0].set_title("Confusion Matrix (Test)"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

# (b) ROC curve
fpr, tpr, _ = roc_curve(y_test, p_test_final)
axes[1].plot(fpr, tpr, color="#3d9970", lw=2,
             label=f"AUC = {roc_auc_score(y_test, p_test_final):.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_title("ROC Curve (Test)"); axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate"); axes[1].legend(loc="lower right")

# (c) Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, p_test_final)
axes[2].plot(rec, prec, color="#c1666b", lw=2,
             label=f"PR-AUC = {average_precision_score(y_test, p_test_final):.4f}")
axes[2].set_title("Precision-Recall Curve (Test)"); axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision"); axes[2].legend(loc="lower left")

plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "final_eval.png"), dpi=120, bbox_inches="tight")
plt.show()

### 19.1 Decision-threshold tuning (optimising for Recall)

Because the brief emphasises **recall** (not missing true millionaires), we examine how precision and
recall trade off across decision thresholds and identify the threshold that maximises F1, as well as a
high-recall operating point.

In [ ]:
ths = np.linspace(0.05, 0.95, 91)
rows = []
for t in ths:
    yp = (p_test_final >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_test, yp, zero_division=0),
                 "recall":    recall_score(y_test, yp, zero_division=0),
                 "f1":        f1_score(y_test, yp, zero_division=0)})
th_df = pd.DataFrame(rows)

best_f1_thr = th_df.loc[th_df["f1"].idxmax(), "threshold"]
best_f1 = th_df["f1"].max()
hi_rec = th_df[th_df["recall"] >= 0.95].sort_values("precision", ascending=False)
hi_rec_thr = hi_rec.iloc[0]["threshold"] if len(hi_rec) else None

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(th_df["threshold"], th_df["precision"], label="Precision", color="#5276a7", lw=2)
ax.plot(th_df["threshold"], th_df["recall"],    label="Recall",    color="#c1666b", lw=2)
ax.plot(th_df["threshold"], th_df["f1"],        label="F1-Score",  color="#3d9970", lw=2)
ax.axvline(0.5, color="gray", ls="--", lw=1, label="Default 0.5")
ax.axvline(best_f1_thr, color="black", ls=":", lw=1.5, label=f"Best F1 thr = {best_f1_thr:.2f}")
ax.set_title("Precision / Recall / F1 vs. Decision Threshold (Test)")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "threshold_tuning.png"), dpi=120, bbox_inches="tight")
plt.show()

print(f"Default threshold 0.50 -> F1={th_df.loc[(th_df.threshold-0.5).abs().idxmin(),'f1']:.4f}")
print(f"Best-F1 threshold {best_f1_thr:.2f} -> F1={best_f1:.4f}")
if hi_rec_thr is not None:
    r = th_df.loc[(th_df.threshold-hi_rec_thr).abs().idxmin()]
    print(f"High-recall threshold {hi_rec_thr:.2f} -> recall={r['recall']:.4f}, precision={r['precision']:.4f}")

## 20. Interpretation — Which Features Drive the Prediction?

We use **permutation importance** (on a test sub-sample for speed) and map the one-hot encoded
columns back to their parent features. This shows which leakage-free personal / lifestyle features
the network relies on, and how the two groups compare.


In [ ]:
class KerasProbaWrapper:
    # Adapter so sklearn's permutation_importance can score the Keras model.
    _estimator_type = "classifier"   # required by sklearn>=1.7 scorer
    def __init__(self, model): self.model = model
    def fit(self, X, y=None): return self
    def predict(self, X): return (self.model.predict(X, verbose=0).ravel() >= 0.5).astype(int)
    def predict_proba(self, X):
        p = self.model.predict(X, verbose=0).ravel()
        return np.vstack([1 - p, p]).T

# sub-sample the test set for a fast, stable estimate
rng = np.random.RandomState(SEED)
sub = rng.choice(len(X_test), size=min(3000, len(X_test)), replace=False)
Xsub, ysub = X_test[sub], y_test[sub]

t0 = time.time()
perm = permutation_importance(KerasProbaWrapper(final_model), Xsub, ysub,
                              scoring="roc_auc", n_repeats=3, random_state=SEED, n_jobs=1)
print(f"Permutation importance computed in {time.time()-t0:.1f}s")

# Map encoded column names back to their parent feature
feat_names = PRE.get_feature_names_out()
known = set(CLEAN_FEATURES)
def to_parent(name):
    if name in known: return name
    # one-hot names look like 'col_value'; find the longest known prefix
    for c in sorted(known, key=len, reverse=True):
        if name.startswith(c + "_"): return c
    return name
parent = np.array([to_parent(n.split("__", 1)[1]) for n in feat_names])

imp = (pd.DataFrame({"feature": parent, "importance": perm.importances_mean})
       .groupby("feature", as_index=False)["importance"].sum()
       .sort_values("importance", ascending=False))
top_imp = imp.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=top_imp[::-1], x="importance", y="feature", ax=ax, color="#5276a7")
ax.set_title("Top 20 Features by Permutation Importance (ROC-AUC drop)")
ax.set_xlabel("Mean ROC-AUC decrease"); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "permutation_importance.png"), dpi=120, bbox_inches="tight")
plt.show()

top_imp.round(5).to_string(index=False)


In [ ]:
# Share of predictive importance by feature group
imp2 = imp.copy()
imp2["group"] = imp2["feature"].map(
    lambda f: next((g for g, cols in FEATURE_GROUPS.items() if f in cols), "Other"))
grp_imp = (imp2.groupby("group")["importance"].sum().sort_values(ascending=False))
share = (grp_imp / grp_imp.sum() * 100).round(1)
print("Share of total permutation importance by feature group (%):")
print(share.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
share.plot(kind="bar", ax=ax, color="#c1666b")
ax.set_title("Share of Predictive Importance by Feature Group")
ax.set_ylabel("% of total importance"); ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "importance_by_group.png"), dpi=120, bbox_inches="tight")
plt.show()


## 21. Conclusions

### Answers to the research questions

**Main RQ — Can a neural network predict millionaire status from personal characteristics and
lifestyle / behavioural habits *alone* (no monetary information)?**
**Only weakly.** After removing *all* monetary and wealth-derived information, the MLP can no longer
recover the ~96% accuracy of the leaky model. Its test performance is close to a majority-class
baseline, with a modest but real lift over chance — the honest signal carried by habits alone.

**RQ1 — How accurately can the network classify the two classes?**
See Section 18 (all experiments) and Section 19 (final evaluation). The leakage-free metrics
(accuracy / precision / recall / F1 / ROC-AUC / PR-AUC) are reported there. Accuracy must be read
alongside ROC-AUC and the confusion matrix, because the classes are ~57/43.

**RQ2 — Which architecture performs best?**
Architecture makes **little difference** once the features are leakage-free: single-layer and deeper
MLPs are statistically close. A small amount of capacity (one hidden layer) is enough to capture the
weak, mostly-monotone signal.

**RQ3 — How do feature groups affect performance?**
* **Personal** features (age, experience, education, network) carry most of the — limited — signal.
* **Lifestyle & Behavioral** habits add little on their own.
* **Combining** both is only marginally better than Personal alone.
See Section 16 (ablation).

**RQ4 — How does tuning affect performance?**
Hyperparameters (learning rate, batch size, depth, dropout, optimiser) change performance only
**marginally**; the honest ceiling is set by the features, not by the network.

### Key takeaways
* **The original ~96% accuracy was financial data leakage.** The dataset contains incomes, savings,
  investment amounts, asset values and financial ratios; a model that can read them is essentially
  reading net worth, not predicting it.
* **Non-financial habits are genuinely predictive only to a small degree.** The strongest
  leakage-free correlates are *age* and *years_of_experience* (r ≈ 0.26) — i.e. simple life-stage
  effects — followed by education and network measures. Most behavioural habit items have almost no
  linear relationship with the target (|r| < 0.02).
* **Honest reporting matters.** We deliberately do **not** force the old 96% number; the value of
  this project is demonstrating *how* to detect and remove leakage and to state the real,
  leakage-free performance.


## 22. Limitations & Notes

1. **Synthetic data.** The dataset is explicitly synthetic (latent-factor generation). Absolute
   numbers will not transfer to real populations; in particular, real-world millionaire status
   *is* strongly driven by income and capital, which we (correctly) refuse to use here.
2. **No billionaires.** The literal `$1B` target is impossible on this data (max net worth
   ≈ \$105.8 M); we therefore model the dataset's native **millionaire** threshold.
3. **Weak honest signal is expected.** Because we removed every monetary column, the remaining
   features can only explain a limited part of wealth — this is the *correct*, non-leaky result,
   not a modelling failure.
4. **Leakage removal is deliberately conservative.** Some retained items (e.g.
   `income_stability_score`, `high_income_peer_exposure`) are *scores* that correlate mildly with
   income; we keep them as behavioural/social signals but flag them for transparency.
5. **Single split.** Results come from one stratified 70/15/15 split with fixed seeds; a repeated /
   cross-validated evaluation would tighten the confidence intervals.
6. **CPU-only training.** Experiments ran on CPU; runtimes are reported per experiment.


## 23. Reproducibility — Save Artefacts

In [ ]:
# Persist the comparison tables, the audit files and the final model
summary.to_csv(os.path.join(OUT_DIR, "all_experiments.csv"), index=False)
headline.to_csv(os.path.join(OUT_DIR, "headline_results.csv"), index=False)
abl.to_csv(os.path.join(OUT_DIR, "feature_group_ablation.csv"), index=False)

# --- leakage-free audit artefacts ---
audit.to_csv(os.path.join(OUT_DIR, "excluded_columns.csv"), index=False)
pd.DataFrame({"feature": CLEAN_FEATURES,
              "group": ["Personal" if c in FEATURE_GROUPS["Personal"] else "Lifestyle & Behavioral"
                        for c in CLEAN_FEATURES],
              "category": [cat_of.get(c, "?") for c in CLEAN_FEATURES],
              "description": [str(data_dict.set_index("column_name")["description"].get(c, ""))
                              for c in CLEAN_FEATURES]}).to_csv(
    os.path.join(OUT_DIR, "retained_features.csv"), index=False)

class_distribution.to_csv(os.path.join(OUT_DIR, "class_distribution.csv"), index=False)

final_model.save(os.path.join(OUT_DIR, "final_mlp_leakage_free.keras"))

print("Saved artefacts to ./outputs:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  -", f)


## 24. Original (Leaky) vs. Leakage-Free Model - Direct Comparison

The **original** numbers below were measured in the *first* version of this notebook, whose
`CLEAN_FEATURES` still contained monetary columns (`annual_salary`, `monthly_income`, `savings_rate`,
`investment_rate`, `passive_income_ratio`, …). They are quoted here **for comparison only**; the
leakage-free model trained above is the honest result.

> The dramatic drop between the two rows is the **leakage effect**: with money visible the network
> reads net worth; without it, only weak habit/life-stage signal remains.


In [ ]:
original_leaky = {
    "version": "Original (monetary features KEPT - leaky)",
    "accuracy": 0.9617, "precision": 0.9618, "recall": 0.9561, "f1": 0.9589,
    "roc_auc": 0.9958, "pr_auc": 0.9945, "tn": 8384, "fp": 242, "fn": 280, "tp": 6094,
}
new_clean = {
    "version": "Leakage-free (personal + lifestyle/behaviour only)",
    "accuracy": final_row["test_accuracy"], "precision": final_row["test_precision"],
    "recall": final_row["test_recall"], "f1": final_row["test_f1"],
    "roc_auc": final_row["test_roc_auc"], "pr_auc": final_row["test_pr_auc"],
    "tn": final_row["test_tn"], "fp": final_row["test_fp"],
    "fn": final_row["test_fn"], "tp": final_row["test_tp"],
}

comparison = pd.DataFrame([original_leaky, new_clean])
comparison_disp = comparison[["version", "accuracy", "precision", "recall", "f1",
                              "roc_auc", "pr_auc"]].copy()
for c in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    comparison_disp[c] = comparison_disp[c].round(4)
print("=== Original (leaky) vs Leakage-free ===")
print(comparison_disp.to_string(index=False))
print("\nConfusion matrices:")
for _, r in comparison.iterrows():
    print(f"  {r['version']:<48s} TN={r['tn']:>6}  FP={r['fp']:>6}  FN={r['fn']:>6}  TP={r['tp']:>6}")

comparison.to_csv(os.path.join(OUT_DIR, "leakage_free_comparison.csv"), index=False)

# grouped bar chart
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
x = np.arange(len(metrics)); w = 0.38
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(x - w/2, [original_leaky[m] for m in metrics], w,
       label="Original (leaky)", color="#c1666b")
ax.bar(x + w/2, [new_clean[m] for m in metrics], w,
       label="Leakage-free", color="#3d9970")
ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in metrics])
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("Original (leaky) vs. Leakage-free Model - Test Metrics")
ax.legend()
for xi, m in zip(x - w/2, metrics):
    ax.annotate(f"{original_leaky[m]:.2f}", (xi, original_leaky[m]), ha="center", va="bottom", fontsize=8)
for xi, m in zip(x + w/2, metrics):
    ax.annotate(f"{new_clean[m]:.2f}", (xi, new_clean[m]), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "leakage_free_comparison.png"), dpi=120, bbox_inches="tight")
plt.show()
